# Camada Gold — ecommerce_pedidos

## Objetivo

Gerar métricas agregadas de Data Quality para consumo analítico e visualização em BI.

### Saídas esperadas

- `gold_dq_resumo_por_regra`
- `gold_dq_resumo_por_tabela`

### Fluxo

`dq_monitoring_logs` + Silver → agregações de qualidade → Gold Delta → Azure SQL

In [0]:
# =========================================================
# IMPORTS
# =========================================================

from pyspark.sql import functions as F

In [0]:
%run ./00_autenticação

In [0]:
# =========================================================
# CONFIGURAÇÃO DOS CAMINHOS
# =========================================================

storage_account = "internshipdatalake"
lake_container = "squad1"

silver_path = (
    f"abfss://{lake_container}@"
    f"{storage_account}.dfs.core.windows.net/"
    f"silver/ecommerce_pedidos"
)

dq_logs_path = (
    f"abfss://{lake_container}@"
    f"{storage_account}.dfs.core.windows.net/"
    f"dq_monitoring_logs"
)

gold_regra_path = (
    f"abfss://{lake_container}@"
    f"{storage_account}.dfs.core.windows.net/"
    f"gold/gold_dq_resumo_por_regra"
)

gold_tabela_path = (
    f"abfss://{lake_container}@"
    f"{storage_account}.dfs.core.windows.net/"
    f"gold/gold_dq_resumo_por_tabela"
)

adls_opts = {
    "fs.azure.account.auth.type": "OAuth",
    "fs.azure.account.oauth.provider.type":
        "org.apache.hadoop.fs.azurebfs.oauth2.ClientCredsTokenProvider",
    "fs.azure.account.oauth2.client.id": client_id,
    "fs.azure.account.oauth2.client.secret": client_secret,
    "fs.azure.account.oauth2.client.endpoint":
        f"https://login.microsoftonline.com/{tenant_id}/oauth2/token"
}

In [0]:
# =========================================================
# LEITURA DAS FONTES
# =========================================================

df_logs = (
    spark.read
    .format("delta")
    .options(**adls_opts)
    .load(dq_logs_path)
)

df_silver = (
    spark.read
    .format("delta")
    .options(**adls_opts)
    .load(silver_path)
)

print("Logs de DQ:", df_logs.count())
print("Registros na Silver:", df_silver.count())

In [0]:
# =========================================================
# GOLD — RESUMO DE DQ POR REGRA
# =========================================================

df_gold_regra = (
    df_logs
    .filter(F.col("tabela") == "ecommerce_pedidos")
    .withColumn(
        "data",
        F.to_date(
            F.regexp_extract(
                "arquivo_origem",
                r"real-time-data/(\d{4}/\d{2}/\d{2})/",
                1
            ),
            "yyyy/MM/dd"
        )
    )
    .groupBy(
        "data",
        "tabela",
        "regra",
        "severidade"
    )
    .agg(
        F.sum("qtd_registros_falhos")
            .alias("qtd_registros_falhos"),

        F.sum("qtd_registros_total")
            .alias("qtd_registros_total")
    )
    .withColumn(
        "pct_falha",
        F.round(
            (
                F.col("qtd_registros_falhos")
                / F.col("qtd_registros_total")
            ) * 100,
            2
        )
    )
    .orderBy(
        "data",
        "regra"
    )
)

df_gold_regra.show(50, truncate=False)

In [0]:
# =========================================================
# VALIDAÇÃO — RESUMO POR REGRA
# =========================================================

print("Linhas na Gold por regra:", df_gold_regra.count())

df_gold_regra.select(
    F.sum(
        F.col("pct_falha").isNull().cast("int")
    ).alias("pct_falha_nulo")
).show()

In [0]:
# =========================================================
# GOLD - RESUMO DE DQ POR TABELA / HORA
# Cálculo linha a linha a partir das flags da Silver
# =========================================================

# Flags de Data Quality da ecommerce_pedidos
dq_flags = [
    "dq_id_pedido_invalido",
    "dq_id_cliente_nulo",
    "dq_status_invalido",
    "dq_valor_total_invalido",
    "dq_metodo_pagamento_invalido",
    "dq_data_status_invalida",
    "dq_frete_invalido",
    "dq_cancelado_com_rastreamento_ativo",
    "dq_entrega_prazo_invalido",
    "dq_endereco_entrega_invalido"
]

# ---------------------------------------------------------
# 1. Avaliação linha a linha
# ---------------------------------------------------------
# greatest retorna True se pelo menos uma flag dq_* for True.
# Assim, um pedido que falha em várias regras continua sendo
# contado somente UMA vez como pedido com falha.
df_gold_tabela_base = (
    df_silver
    .withColumn(
        "_tem_falha",
        F.greatest(
            *[
                F.coalesce(F.col(flag), F.lit(False))
                for flag in dq_flags
            ]
        )
    )
    .withColumn(
        "_data",
        F.to_date(
            F.regexp_extract(
                "bronze_source_file",
                r"real-time-data/(\d{4}/\d{2}/\d{2})/",
                1
            ),
            "yyyy/MM/dd"
        )
    )
    .withColumn(
        "_hora",
        F.regexp_extract(
            "bronze_source_file",
            r"real-time-data/\d{4}/\d{2}/\d{2}/(\d{2})\d{4}/",
            1
        ).cast("int")
    )
)

# ---------------------------------------------------------
# 2. Agregação da Gold
# ---------------------------------------------------------
df_gold_tabela = (
    df_gold_tabela_base
    .groupBy("_data", "_hora")
    .agg(
        F.sum(
            F.when(
                F.col("_tem_falha") == False,
                1
            ).otherwise(0)
        ).alias("qtd_limpos"),

        F.count("*").alias("qtd_total")
    )
    .withColumn(
        "tabela",
        F.lit("ecommerce_pedidos")
    )
    .withColumn(
        "pct_limpo",
        F.round(
            (
                F.col("qtd_limpos")
                / F.col("qtd_total")
            ) * 100,
            2
        )
    )
    .select(
        "_data",
        "_hora",
        "qtd_limpos",
        "qtd_total",
        "tabela",
        "pct_limpo"
    )
    .orderBy("_data", "_hora")
)

display(df_gold_tabela)

In [0]:
# =========================================================
# VALIDAÇÃO - GOLD POR TABELA
# =========================================================

validacao = (
    df_gold_tabela
    .agg(
        F.sum("qtd_total").alias("total_registros"),
        F.sum("qtd_limpos").alias("total_limpos")
    )
    .withColumn(
        "total_com_falha",
        F.col("total_registros") - F.col("total_limpos")
    )
    .withColumn(
        "pct_limpo_global",
        F.round(
            (F.col("total_limpos") / F.col("total_registros")) * 100,
            2
        )
    )
)

display(validacao)

In [0]:
# =========================================================
# ADAPTAÇÃO PARA O DELTA COMPARTILHADO
# =========================================================

# Mantém as métricas antigas do contrato compartilhado,
# mas usa o percentual de registros limpos calculado
# corretamente linha a linha na Silver.

df_metricas_logs_hora = (
    df_logs
    .filter(F.col("tabela") == "ecommerce_pedidos")
    .withColumn(
        "hora",
        F.to_timestamp(
            F.regexp_extract(
                "arquivo_origem",
                r"real-time-data/(\d{4}/\d{2}/\d{2}/\d{2})\d{4}/",
                1
            ),
            "yyyy/MM/dd/HH"
        )
    )
    .groupBy("hora", "tabela")
    .agg(
        F.countDistinct("regra").alias("qtd_regras_avaliadas"),

        F.countDistinct(
            F.when(
                F.col("qtd_registros_falhos") > 0,
                F.col("regra")
            )
        ).alias("qtd_regras_com_falha"),

        F.sum("qtd_registros_falhos").alias("soma_falhas")
    )
)

df_gold_tabela_delta = (
    df_gold_tabela
    .withColumn(
        "hora",
        F.to_timestamp(
            F.concat(
                F.col("_data").cast("string"),
                F.lit(" "),
                F.lpad(F.col("_hora").cast("string"), 2, "0"),
                F.lit(":00:00")
            )
        )
    )
    .join(
        df_metricas_logs_hora,
        on=["hora", "tabela"],
        how="left"
    )
    .withColumn("soma_total", F.col("qtd_total"))
    .withColumn("pct_registros_limpos", F.col("pct_limpo"))
    .withColumn("gold_processed_at", F.current_timestamp())
    .select(
        "hora",
        "tabela",
        "qtd_regras_avaliadas",
        "qtd_regras_com_falha",
        "soma_falhas",
        "soma_total",
        "pct_registros_limpos",
        "gold_processed_at"
    )
    .orderBy("hora")
)

display(df_gold_tabela_delta)

In [0]:
# =========================================================
# PERSISTÊNCIA DAS GOLDS NA DELTA
# Atualiza somente ecommerce_pedidos nas tabelas compartilhadas
# =========================================================

# Gold por regra
(
    df_gold_regra
    .write
    .format("delta")
    .mode("overwrite")
    .option("replaceWhere", "tabela = 'ecommerce_pedidos'")
    .options(**adls_opts)
    .save(gold_regra_path)
)

print("OK: Gold por regra atualizada na Delta.")

# Gold por tabela
# Usa a versão adaptada ao schema compartilhado do Delta
(
    df_gold_tabela_delta
    .write
    .format("delta")
    .mode("overwrite")
    .option("replaceWhere", "tabela = 'ecommerce_pedidos'")
    .options(**adls_opts)
    .save(gold_tabela_path)
)

print("OK: Gold por tabela atualizada na Delta.")

In [0]:
# =========================================================
# VALIDAÇÃO DA PERSISTÊNCIA DELTA
# =========================================================

# ---------- Gold por regra ----------
df_gold_regra_salva = (
    spark.read
    .format("delta")
    .options(**adls_opts)
    .load(gold_regra_path)
    .filter(F.col("tabela") == "ecommerce_pedidos")
)

# ---------- Gold por tabela ----------
df_gold_tabela_salva = (
    spark.read
    .format("delta")
    .options(**adls_opts)
    .load(gold_tabela_path)
    .filter(F.col("tabela") == "ecommerce_pedidos")
)

total_gold_tabela = (
    df_gold_tabela_salva
    .agg(F.sum("soma_total").alias("total"))
    .first()["total"]
)

regras_distintas = (
    df_gold_regra_salva
    .select("regra")
    .distinct()
    .count()
)

pct_regra_nulos = (
    df_gold_regra_salva
    .filter(F.col("pct_falha").isNull())
    .count()
)

pct_tabela_nulos = (
    df_gold_tabela_salva
    .filter(F.col("pct_registros_limpos").isNull())
    .count()
)

print("Linhas Gold por regra:", df_gold_regra_salva.count())
print("Regras distintas:", regras_distintas)

print("Linhas Gold por tabela:", df_gold_tabela_salva.count())
print("Total representado na Gold por tabela:", total_gold_tabela)

print("Pct falha nulos:", pct_regra_nulos)
print("Pct registros limpos nulos:", pct_tabela_nulos)

assert regras_distintas == 10, \
    "A Gold por regra não contém as 10 regras esperadas."

assert total_gold_tabela == df_silver.count(), \
    "A Gold por tabela não representa o mesmo total da Silver."

assert pct_regra_nulos == 0, \
    "Existem percentuais de falha nulos."

assert pct_tabela_nulos == 0, \
    "Existem percentuais de registros limpos nulos."

print("OK: Golds Delta validadas.")

In [0]:
# =========================================================
# PERSISTÊNCIA DAS GOLDS DA DUPLA 4 NO AZURE SQL
# =========================================================

sql_gold_regra = "squad1.dupla4_ecommerce_pedidos_gold_dq_regra"
sql_gold_tabela = "squad1.dupla4_ecommerce_pedidos_gold_dq_tabela"

(
    df_gold_regra
    .write
    .format("sqlserver")
    .mode("overwrite")
    .option("host", jdbc_hostname)
    .option("database", jdbc_database)
    .option("user", jdbc_username)
    .option("password", jdbc_password)
    .option("authentication", "SqlPassword")
    .option("encrypt", "true")
    .option("dbtable", sql_gold_regra)
    .save()
)

(
    df_gold_tabela
    .write
    .format("sqlserver")
    .mode("overwrite")
    .option("host", jdbc_hostname)
    .option("database", jdbc_database)
    .option("user", jdbc_username)
    .option("password", jdbc_password)
    .option("authentication", "SqlPassword")
    .option("encrypt", "true")
    .option("dbtable", sql_gold_tabela)
    .save()
)

print("OK: Golds de ecommerce_pedidos da Dupla 4 gravadas no Azure SQL.")

In [0]:
# =========================================================
# VALIDAÇÃO FINAL DAS GOLDS NO AZURE SQL
# =========================================================

df_sql_gold_regra = spark.read.jdbc(
    url=jdbc_url,
    table=sql_gold_regra,
    properties=connection_properties
)

df_sql_gold_tabela = spark.read.jdbc(
    url=jdbc_url,
    table=sql_gold_tabela,
    properties=connection_properties
)

qtd_regra_esperada = df_gold_regra.count()
qtd_tabela_esperada = df_gold_tabela.count()

qtd_regra_sql = df_sql_gold_regra.count()
qtd_tabela_sql = df_sql_gold_tabela.count()

print(f"Gold por regra  | esperado: {qtd_regra_esperada} | SQL: {qtd_regra_sql}")
print(f"Gold por tabela | esperado: {qtd_tabela_esperada} | SQL: {qtd_tabela_sql}")

assert qtd_regra_sql == qtd_regra_esperada, "Divergência na Gold por regra."
assert qtd_tabela_sql == qtd_tabela_esperada, "Divergência na Gold por tabela."

print("OK: Golds no Azure SQL validadas com sucesso.")

## Observações finais

- A Gold por regra foi construída a partir dos logs de Data Quality, consolidando os resultados das regras avaliadas para `ecommerce_pedidos`.
- A Gold por tabela foi calculada **linha a linha a partir da Silver**, considerando um pedido com falha quando qualquer flag `dq_*` estiver verdadeira.
- Dessa forma, um mesmo pedido é contado apenas uma vez como registro com falha, mesmo quando viola mais de uma regra.
- O percentual de registros limpos é calculado a partir da quantidade de registros sem nenhuma falha em relação ao total processado.
- No Delta compartilhado da Squad 1, foi mantido o schema existente por compatibilidade, utilizando o percentual correto calculado linha a linha.
- No Azure SQL, a Gold por tabela utiliza o contrato destinado ao consumo analítico: `_data`, `_hora`, `qtd_limpos`, `qtd_total`, `tabela` e `pct_limpo`.
- As gravações no Delta utilizam `replaceWhere` para atualizar somente os registros de `ecommerce_pedidos`, preservando as demais tabelas compartilhadas.
- As Golds são validadas após a persistência tanto no Delta quanto no Azure SQL.